# 05. TabPFN 버전 비교 (TabPFN 계열만 사용)

`04`에서 고른 **변수 조합 · 타깃 변환은 그대로** 두고, **TabPFN 버전만 바꿔서** 같은 CV로 비교한다.

| 단계 | 내용 |
| --- | --- |
| 1 | 04 결과에서 최고 조합(변수 그룹 · 타깃) 불러오기 |
| 2 | 버전별 CV: v2 · 2.5 · 2.6 · 3 · 3.5 · 3.5-fast |
| 3 | (선택) TabPFN 버전끼리 예측 평균 |
| 4 | 리더보드 확인용 제출 파일: 최고 버전 × (04 타깃 / log 타깃) |

- 검증: 팀 공통 5-Fold (seed 42), 원래 관객수 단위 RMSE
- 버전마다 처음 실행할 때 모델 가중치를 내려받음 (라이선스 동의 + `TABPFN_TOKEN` 필요, v2는 불필요)
- 결과는 `cache_05/`에 저장 → 다시 실행하면 건너뜀
- 파생변수 함수는 `features.py` (04와 같은 코드)

## 0. 설정

In [1]:
import os
import glob
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn import set_config
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from features import make_features

warnings.filterwarnings('ignore')
set_config(display='text')

DATA_DIR = '.'
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE_DIR = 'cache_05'
os.makedirs(CACHE_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
y = train['box_off_num']
hit = y >= 1_000_000

TRANSFORMS = {
    'raw':  (lambda v: v, lambda p: p),
    'log':  (np.log1p,    np.expm1),
    'sqrt': (np.sqrt,     lambda p: np.square(np.clip(p, 0, None))),
}
print('device:', DEVICE)

device: cpu


C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 04 최고 조합 불러오기

In [2]:
res04 = [pickle.load(open(p, 'rb')) for p in glob.glob('cache_04/*.pkl')]
best04 = min(res04, key=lambda r: r['CV RMSE'])
OPTS, TARGET_T = best04['opts'], best04['타깃']

X, X_test = make_features(train, test, OPTS)
print('04 최고 실험:', best04['실험'])
print(f"  CV RMSE {best04['CV RMSE']:,.0f} | 변수 그룹 {[k for k in OPTS if OPTS[k]]} | 타깃 {TARGET_T} | 변수 {X.shape[1]}개")

04 최고 실험: C_debut+distributor+date+rating_sqrt
  CV RMSE 1,357,181 | 변수 그룹 ['debut', 'distributor', 'date', 'rating'] | 타깃 sqrt | 변수 42개


## 2. 버전별 CV
버전마다 `create_default_for_version`으로 그 버전의 기본 설정을 씀.
가중치 다운로드나 라이선스 문제로 실패한 버전은 건너뛰고 표에 표시.

In [3]:
VERSIONS = {
    'v2': ModelVersion.V2,
    'v2.5': ModelVersion.V2_5,
    'v2.6': ModelVersion.V2_6,
    'v3': ModelVersion.V3,
    'v3.5': ModelVersion.V3_5,
    'v3.5-fast': ModelVersion.V3_5_FAST,
}


def make_model(version):
    return TabPFNRegressor.create_default_for_version(VERSIONS[version], device=DEVICE, random_state=SEED)


def run_cv(version, target):
    path = os.path.join(CACHE_DIR, f'{version}_{target}.pkl')
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    fwd, inv = TRANSFORMS[target]
    oof = np.zeros(len(y))
    fold_rmse = []
    t0 = time.time()
    for tr_idx, va_idx in kf.split(X):
        model = make_model(version)
        model.fit(X.iloc[tr_idx], fwd(y.iloc[tr_idx]))
        oof[va_idx] = np.clip(inv(model.predict(X.iloc[va_idx])), 0, None)
        fold_rmse.append(mean_squared_error(y.iloc[va_idx], oof[va_idx]) ** 0.5)
    res = {'버전': version, '타깃': target,
           'CV RMSE': mean_squared_error(y, oof) ** 0.5,
           'fold 표준편차': np.std(fold_rmse),
           '흥행작 RMSE': mean_squared_error(y[hit], oof[hit]) ** 0.5,
           '흥행작 외 RMSE': mean_squared_error(y[~hit], oof[~hit]) ** 0.5,
           '소요(초)': time.time() - t0, 'oof': oof}
    pickle.dump(res, open(path, 'wb'))
    return res


results = {}
for v in VERSIONS:
    try:
        results[v] = run_cv(v, TARGET_T)
        r = results[v]
        print(f"{v:<10} CV RMSE {r['CV RMSE']:>11,.0f} | fold std {r['fold 표준편차']:>9,.0f} | 흥행작 {r['흥행작 RMSE']:>11,.0f} | {r['소요(초)']:.0f}초")
    except Exception as e:
        print(f'{v:<10} 실패: {type(e).__name__}: {str(e)[:150]}')

v2         CV RMSE   1,375,455 | fold std   317,730 | 흥행작   3,094,747 | 164초


v2.5       CV RMSE   1,374,933 | fold std   317,797 | 흥행작   3,102,200 | 191초


v2.6       CV RMSE   1,349,691 | fold std   339,162 | 흥행작   3,035,369 | 601초


v3         CV RMSE   1,394,592 | fold std   313,420 | 흥행작   3,122,828 | 277초


v3.5       CV RMSE   1,357,181 | fold std   320,882 | 흥행작   3,051,397 | 387초


v3.5-fast  CV RMSE   1,382,388 | fold std   324,609 | 흥행작   3,117,136 | 168초


In [4]:
FMT = {'CV RMSE': '{:,.0f}', 'fold 표준편차': '{:,.0f}', '흥행작 RMSE': '{:,.0f}', '흥행작 외 RMSE': '{:,.0f}', '소요(초)': '{:.0f}'}
table = pd.DataFrame([{k: v for k, v in r.items() if k != 'oof'} for r in results.values()]).sort_values('CV RMSE')
display(table.style.format(FMT).highlight_min(subset=['CV RMSE', '흥행작 RMSE', '흥행작 외 RMSE'], color='#c6efce'))

best_v = table.iloc[0]['버전']
print('→ CV 최고 버전:', best_v)

,버전,타깃,CV RMSE,fold 표준편차,흥행작 RMSE,흥행작 외 RMSE,소요(초)
2,v2.6,sqrt,"1,349,691","339,162","3,035,369","425,109",601
4,v3.5,sqrt,"1,357,181","320,882","3,051,397","428,757",387
1,v2.5,sqrt,"1,374,933","317,797","3,102,200","416,773",191
0,v2,sqrt,"1,375,455","317,730","3,094,747","430,940",164
5,v3.5-fast,sqrt,"1,382,388","324,609","3,117,136","422,136",168
3,v3,sqrt,"1,394,592","313,420","3,122,828","460,137",277


→ CV 최고 버전: v2.6


## 3. (선택) TabPFN 버전끼리 예측 평균
다른 계열 모델은 섞지 않고, **TabPFN 버전들의 예측만** 평균. CV 상위 k개 버전 평균을 비교.

In [5]:
ranked = list(table['버전'])
rows = []
for k in range(1, len(ranked) + 1):
    avg = np.mean([results[v]['oof'] for v in ranked[:k]], axis=0)
    rows.append({'평균한 버전 수': k, '버전': ', '.join(ranked[:k]),
                 'CV RMSE': mean_squared_error(y, avg) ** 0.5,
                 '흥행작 RMSE': mean_squared_error(y[hit], avg[hit]) ** 0.5})
avg_df = pd.DataFrame(rows)
display(avg_df.style.format({'CV RMSE': '{:,.0f}', '흥행작 RMSE': '{:,.0f}'}).highlight_min(subset=['CV RMSE'], color='#c6efce'))

best_k = int(avg_df.loc[avg_df['CV RMSE'].idxmin(), '평균한 버전 수'])
USE_AVG = best_k > 1 and avg_df['CV RMSE'].min() < table.iloc[0]['CV RMSE'] * 0.99   # 1% 이상 좋아질 때만
print(f'버전 평균 최고: 상위 {best_k}개 → 사용: {USE_AVG}')

,평균한 버전 수,버전,CV RMSE,흥행작 RMSE
0,1,v2.6,"1,349,691","3,035,369"
1,2,"v2.6, v3.5","1,348,773","3,034,125"
2,3,"v2.6, v3.5, v2.5","1,355,126","3,051,735"
3,4,"v2.6, v3.5, v2.5, v2","1,358,450","3,058,659"
4,5,"v2.6, v3.5, v2.5, v2, v3.5-fast","1,361,134","3,066,084"
5,6,"v2.6, v3.5, v2.5, v2, v3.5-fast, v3","1,365,528","3,073,407"


버전 평균 최고: 상위 2개 → 사용: False


## 4. 리더보드 확인용 제출 파일
- **CV 기준 최고**: 최고 버전 × 04 타깃(`TARGET_T`) (+ 버전 평균이 좋으면 그 파일도)
- **리더보드 가설 확인용**: 최고 버전 × **log 타깃**
  → 코드 공유 1위(Public 493,643)가 log 타깃이었음. Public 리더보드에 초대박 영화가 적다면 log처럼 **보수적인 예측이 리더보드에서 유리**할 수 있음.
  → 두 파일을 같이 제출해서 비교해보기

In [6]:
def fit_predict(version, target):
    fwd, inv = TRANSFORMS[target]
    model = make_model(version)
    model.fit(X, fwd(y))
    return np.clip(inv(model.predict(X_test)), 0, None)


def save(pred, name):
    sub = submission.copy()
    sub['box_off_num'] = pred
    sub.to_csv(name, index=False)
    print(f'저장: {name} | 예측 평균 {pred.mean():,.0f} · 최대 {pred.max():,.0f}')
    return sub


preds = {}
preds[TARGET_T] = fit_predict(best_v, TARGET_T)
save(preds[TARGET_T], f'05_{best_v}_{TARGET_T}.csv')

if TARGET_T != 'log':
    preds['log'] = fit_predict(best_v, 'log')
    save(preds['log'], f'05_{best_v}_log.csv')

if USE_AVG:
    avg_pred = np.mean([fit_predict(v, TARGET_T) for v in ranked[:best_k]], axis=0)
    save(avg_pred, f'05_avg_top{best_k}_{TARGET_T}.csv')

저장: 05_v2.6_sqrt.csv | 예측 평균 618,822 · 최대 7,661,487


저장: 05_v2.6_log.csv | 예측 평균 485,250 · 최대 6,457,168


In [7]:
# 두 타깃의 test 예측이 어떻게 다른지: 예측 관객수 상위 10편
cmp = pd.DataFrame({'제목': test['title']} | {f'{best_v} {t}': p for t, p in preds.items()})
display(cmp.sort_values(f'{best_v} {TARGET_T}', ascending=False).head(10)
        .style.format({c: '{:,.0f}' for c in cmp.columns if c != '제목'}))

,제목,v2.6 sqrt,v2.6 log
135,용의자,"7,661,487","6,457,168"
178,명량,"7,438,489","5,897,084"
179,군도: 민란의 시대,"6,600,234","5,613,277"
142,박수건달,"5,732,638","4,750,600"
229,베테랑,"4,423,353","3,375,149"
106,댄싱퀸,"3,955,222","2,850,186"
108,신세계,"3,629,860","3,127,566"
134,동창생,"3,437,566","3,128,243"
83,늑대소년,"3,338,072","2,623,756"
141,마이 리틀 히어로,"3,273,334","1,996,468"


In [8]:
table.drop(columns=[]).to_csv('05_results.csv', index=False, encoding='utf-8-sig')
print('05_results.csv 저장 (노션 리더보드 기록용)')

05_results.csv 저장 (노션 리더보드 기록용)
